<a href="https://colab.research.google.com/github/yoolmalaak/concrete-crack-and-fracture-analysis/blob/main/notebooks/08_unet_crack_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08. U-Net Crack Segmentation

This notebook develops a U-Net-based image segmentation model for concrete crack analysis. The trained segmentation model will be evaluated using ground-truth crack masks and subsequently applied to selected images from SDNET2018 for crack morphology extraction.

## 1. Environment Setup

### 1.1 Install Dependencies

In [ ]:
!pip install -q datasets

In [ ]:
import datasets

print("datasets version:", datasets.__version__)

### 1.2 Import Libraries

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from PIL import Image

### 1.3 Set Random Seeds

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed set to:", SEED)
print("CUDA available:", torch.cuda.is_available())

Random seed set to: 42
CUDA available: False


## 2. Load CrackSeg9k Dataset

### 2.1 Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "rimvydasrub/crackseg9k",
    trust_remote_code=False
)

print(dataset)

### 2.2 Inspect Dataset Structure

In [ ]:
print("Train samples:", len(dataset["train"]))
print("Test samples:", len(dataset["test"]))

print("\nFeatures:")
print(dataset["train"].features)

print("\nFirst training sample:")
print(dataset["train"][0])

### 2.3 Check Image and Mask Fields

In [ ]:
import base64
from io import BytesIO

sample = dataset["train"][0]

image = Image.open(BytesIO(base64.b64decode(sample["image"]))).convert("RGB")
mask = Image.open(BytesIO(base64.b64decode(sample["mask"])))

print("Image size:", image.size)
print("Image mode:", image.mode)

print("Mask size:", mask.size)
print("Mask mode:", mask.mode)

## 3. Dataset Inspection

### 3.1 Display Sample Image

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title("Sample Crack Image")
plt.show()

### 3.2 Display Corresponding Ground-Truth Mask

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(mask, cmap="gray")
plt.axis("off")
plt.title("Ground-Truth Crack Mask")
plt.show()

### 3.3 Overlay Image and Mask

In [ ]:
plt.figure(figsize=(6, 6))

plt.imshow(image)
plt.imshow(mask, cmap="Reds", alpha=0.5)

plt.axis("off")
plt.title("Crack Image with Ground-Truth Mask Overlay")
plt.show()

## 4. Data Preparation

### 4.1 Define Image and Mask Size

In [ ]:
IMAGE_SIZE = (256, 256)

print("Target image size:", IMAGE_SIZE)
print("Input channels: 3")
print("Output channels: 1")

### 4.2 Define Image and Mask Transformations

In [ ]:
from torchvision import transforms
from torchvision.transforms import InterpolationMode

image_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE, interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

mask_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE, interpolation=InterpolationMode.NEAREST),
    transforms.ToTensor()
])

print("Image and mask transformations defined.")

### 4.3 Create PyTorch Dataset

In [ ]:
class CrackSegmentationDataset(Dataset):
    def __init__(self, hf_dataset, image_transform=None, mask_transform=None):
        self.dataset = hf_dataset
        self.image_transform = image_transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        image = Image.open(
            BytesIO(base64.b64decode(sample["image"]))
        ).convert("RGB")

        mask = Image.open(
            BytesIO(base64.b64decode(sample["mask"]))
        ).convert("L")

        if self.image_transform:
            image = self.image_transform(image)

        if self.mask_transform:
            mask = self.mask_transform(mask)

        mask = (mask > 0).float()

        return image, mask

In [ ]:
train_dataset = CrackSegmentationDataset(
    dataset["train"],
    image_transform=image_transform,
    mask_transform=mask_transform
)

test_dataset = CrackSegmentationDataset(
    dataset["test"],
    image_transform=image_transform,
    mask_transform=mask_transform
)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

### 4.4 Create Train/Validation/Test Splits

In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

print("Training samples:", len(train_subset))
print("Validation samples:", len(val_subset))
print("Test samples:", len(test_dataset))

### 4.5 Create DataLoaders

In [ ]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Batch size:", BATCH_SIZE)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

## 5. U-Net Architecture

### 5.1 Double Conv Block

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

### 5.2 Downsampling Block

In [ ]:
class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        features = self.conv(x)
        pooled = self.pool(features)

        return features, pooled

### 5.3 Upsampling Block

In [ ]:
class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            in_channels // 2,
            kernel_size=2,
            stride=2
        )

        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)

        x = torch.cat([skip, x], dim=1)

        x = self.conv(x)

        return x

### 5.4 U-Net Model

In [ ]:
class UNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = DoubleConv(3, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(512, 1024)

        self.up4 = UpBlock(1024, 512)
        self.up3 = UpBlock(512, 256)
        self.up2 = UpBlock(256, 128)
        self.up1 = UpBlock(128, 64)

        self.output = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b, e4)
        d3 = self.up3(d4, e3)
        d2 = self.up2(d3, e2)
        d1 = self.up1(d2, e1)

        return self.output(d1)

### 5.5 Initialize and Test U-Net

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(DEVICE)

sample_batch = next(iter(train_loader))
sample_images, sample_masks = sample_batch

sample_images = sample_images.to(DEVICE)

with torch.no_grad():
    sample_output = model(sample_images)

print("Device:", DEVICE)
print("Input shape:", sample_images.shape)
print("Output shape:", sample_output.shape)

## 6. Training Configuration

### 6.1 Loss Function

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, predictions, targets):
        predictions = torch.sigmoid(predictions)

        predictions = predictions.view(predictions.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        intersection = (predictions * targets).sum(dim=1)

        dice = (
            (2.0 * intersection + self.smooth)
            / (
                predictions.sum(dim=1)
                + targets.sum(dim=1)
                + self.smooth
            )
        )

        return 1 - dice.mean()


bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()


def combined_loss(predictions, targets):
    return bce_loss(predictions, targets) + dice_loss(predictions, targets)


print("BCE + Dice loss defined.")

### 6.2 Evaluation Metrics

In [ ]:
def dice_coefficient(predictions, targets, threshold=0.5, smooth=1.0):
    predictions = torch.sigmoid(predictions)
    predictions = (predictions > threshold).float()

    predictions = predictions.view(predictions.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (predictions * targets).sum(dim=1)

    dice = (
        (2.0 * intersection + smooth)
        / (
            predictions.sum(dim=1)
            + targets.sum(dim=1)
            + smooth
        )
    )

    return dice.mean().item()


def iou_score(predictions, targets, threshold=0.5, smooth=1.0):
    predictions = torch.sigmoid(predictions)
    predictions = (predictions > threshold).float()

    predictions = predictions.view(predictions.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (predictions * targets).sum(dim=1)
    union = predictions.sum(dim=1) + targets.sum(dim=1) - intersection

    iou = (intersection + smooth) / (union + smooth)

    return iou.mean().item()


print("Dice and IoU metrics defined.")

### 6.3 Optimizer

In [ ]:
LEARNING_RATE = 1e-4

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print("Optimizer: Adam")
print("Learning rate:", LEARNING_RATE)

### 6.4 Training Configuration

In [ ]:
NUM_EPOCHS = 20

best_val_dice = 0.0
best_epoch = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "train_dice": [],
    "val_dice": [],
    "train_iou": [],
    "val_iou": []
}

print("Epochs:", NUM_EPOCHS)
print("Best-model criterion: Validation Dice")

## 7. Train U-Net

### 7.1 Training Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0

    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = combined_loss(outputs, masks)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_dice += dice_coefficient(outputs.detach(), masks)
        total_iou += iou_score(outputs.detach(), masks)

    return (
        total_loss / len(loader),
        total_dice / len(loader),
        total_iou / len(loader)
    )


def validate_one_epoch(model, loader, device):
    model.eval()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            loss = combined_loss(outputs, masks)

            total_loss += loss.item()
            total_dice += dice_coefficient(outputs, masks)
            total_iou += iou_score(outputs, masks)

    return (
        total_loss / len(loader),
        total_dice / len(loader),
        total_iou / len(loader)
    )

### 7.2 Train Model

In [ ]:
from pathlib import Path
import time

MODEL_DIR = PROJECT_ROOT / "trained_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for epoch in range(NUM_EPOCHS):
    start_time = time.time()

    train_loss, train_dice, train_iou = train_one_epoch(
        model,
        train_loader,
        optimizer,
        DEVICE
    )

    val_loss, val_dice, val_iou = validate_one_epoch(
        model,
        val_loader,
        DEVICE
    )

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_dice"].append(train_dice)
    history["val_dice"].append(val_dice)
    history["train_iou"].append(train_iou)
    history["val_iou"].append(val_iou)

    elapsed = time.time() - start_time

    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Train Dice: {train_dice:.4f} | "
        f"Val Dice: {val_dice:.4f} | "
        f"Train IoU: {train_iou:.4f} | "
        f"Val IoU: {val_iou:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        best_epoch = epoch + 1

        torch.save(
            {
                "epoch": best_epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "val_dice": val_dice,
                "val_iou": val_iou
            },
            MODEL_DIR / "best_unet.pt"
        )

        print(f"Best model saved at epoch {best_epoch}.")